# 08 - Gold tables for analytics and Genie

Builds two Unity Catalog gold tables so the Genie space (`07_genie_space`) can answer questions about **model scores**, not just raw events:

* **`gold_scored_transfers`**: every silver transfer scored with the model version currently served by `transfer-fraud-model`. Scoring uses `fe.score_batch` with point-in-time feature lookups, so each transfer gets the features it had at transfer time. The table also stores the review-queue flag (top `flag_top_pct`% of scores, default 2%) and the threshold used. Keyed by `sequence_number`, because live generator runs reuse the fixed seed, so a `transfer_id` can repeat at different times.
* **`gold_daily_fraud_summary`**: one row per day, corridor and payout method, with transfers, fraud, flagged, flagged-and-fraud counts and amounts.

Notes:
* The served model was refit on all transfers, so these scores are **in-sample** for historical rows. Holdout metrics are in `05_train_model`.
* Analyst decisions (approve/reject) live in the Lakebase queue (`06_review_queue`) and are served by the app; they are not copied here.
* No PII columns (device_id, ip, phone) are carried into gold.

In [0]:
%pip install -q "databricks-feature-engineering>=0.17" lightgbm "psycopg[binary]" "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks.feature_engineering import FeatureEngineeringClient

dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
dbutils.widgets.text("flag_top_pct", "2")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
FLAG_TOP_PCT = float(dbutils.widgets.get("flag_top_pct"))
C = f"{CATALOG}.{SCHEMA}"
MODEL_NAME = f"{C}.transfer_fraud_model"
ENDPOINT = "transfer-fraud-model"

w = WorkspaceClient()
fe = FeatureEngineeringClient()
served = [e for e in w.serving_endpoints.get(ENDPOINT).config.served_entities if e.entity_name == MODEL_NAME]
assert len(served) == 1, served
MODEL_VERSION = served[0].entity_version
MODEL_URI = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
print(MODEL_URI, "| flag top", FLAG_TOP_PCT, "%")

In [0]:
transfers = (spark.table(f"{C}.silver_transfers")
             .select("sequence_number", "transfer_id", "customer_id", "corridor", "ts", F.col("ts").alias("event_ts"),
                     "payout_method", "amount_usd", "is_fraud"))
scored = fe.score_batch(model_uri=MODEL_URI, df=transfers, result_type="double")
spd = scored.select("sequence_number", "transfer_id", "customer_id", "corridor", "payout_method", "amount_usd", "ts",
                    "is_fraud", F.col("prediction").alias("fraud_score")).toPandas()
assert spd["fraud_score"].between(0, 1).all() and spd["fraud_score"].nunique() > 2, "expected probabilities"

THRESHOLD = float(spd["fraud_score"].quantile(1 - FLAG_TOP_PCT / 100))
# Strict '>' on purpose: transfers with no activity in any feature window get identical feature vectors and
# therefore identical scores (e.g. 529 cash_pickup transfers share one score). When the 98th percentile lands on such
# a tie, '>=' would sweep the whole tie group in and flag ~3.5% instead of ~2%.
spd["flagged"] = (spd["fraud_score"] > THRESHOLD).astype(int)
print(f"transfers tied exactly at the threshold (not flagged): {int((spd['fraud_score'] == THRESHOLD).sum())}")
spd["score_threshold"] = THRESHOLD
spd["model_name"], spd["model_version"] = MODEL_NAME, str(MODEL_VERSION)

gold = (spark.createDataFrame(spd)
        .withColumn("amount_usd", F.col("amount_usd").cast("decimal(12,2)"))
        .withColumn("ts", F.col("ts").cast("timestamp"))
        .withColumn("scored_at", F.current_timestamp()))
gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{C}.gold_scored_transfers")
print(f"wrote {len(spd)} rows | threshold={THRESHOLD:.4f} | flagged={int(spd.flagged.sum())}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {C}.gold_daily_fraud_summary AS
SELECT DATE(ts)                                                      AS day,
       corridor,
       payout_method,
       COUNT(*)                                                      AS transfers,
       SUM(is_fraud)                                                 AS fraud_transfers,
       SUM(flagged)                                                  AS flagged_transfers,
       SUM(CASE WHEN flagged = 1 AND is_fraud = 1 THEN 1 ELSE 0 END) AS flagged_fraud_transfers,
       CAST(SUM(amount_usd) AS DECIMAL(14,2))                        AS total_amount_usd,
       CAST(SUM(CASE WHEN is_fraud = 1 THEN amount_usd ELSE 0 END) AS DECIMAL(14,2)) AS fraud_amount_usd,
       ROUND(AVG(fraud_score), 6)                                    AS avg_fraud_score
FROM {C}.gold_scored_transfers
GROUP BY DATE(ts), corridor, payout_method
""")
display(spark.sql(f"SELECT * FROM {C}.gold_daily_fraud_summary ORDER BY day DESC, fraud_transfers DESC LIMIT 10"))

In [0]:
P = "fraud_rt_"  # project tag prefix (this workspace has governed tag policies on generic keys)
T1, T2 = f"{C}.gold_scored_transfers", f"{C}.gold_daily_fraud_summary"
spark.sql(f"COMMENT ON TABLE {T1} IS 'Gold: every silver transfer scored by transfer_fraud_model (served version) with point-in-time features; flagged = top {FLAG_TOP_PCT:g}% of scores. In-sample for history (model refit on all data). Synthetic data, no PII.'")
spark.sql(f"COMMENT ON TABLE {T2} IS 'Gold: daily transfers, fraud, flagged and flagged-fraud counts and amounts by corridor and payout method, from gold_scored_transfers. Synthetic data.'")
comments = {
    "sequence_number": "Kinesis sequence number (unique key; transfer_id can repeat across live runs)",
    "fraud_score": "Model probability that the transfer is fraud (0-1)",
    "flagged": "1 if fraud_score > score_threshold (top flag_top_pct of scores; ties at the threshold are not flagged)",
    "score_threshold": "Score cut-off for the top flag_top_pct of transfers",
    "is_fraud": "Synthetic ground-truth label (1 = fraud)",
    "model_version": "Unity Catalog model version used for scoring",
}
for c, txt in comments.items():
    spark.sql(f"ALTER TABLE {T1} ALTER COLUMN {c} COMMENT '{txt}'")
for t in (T1, T2):
    spark.sql(f"ALTER TABLE {t} SET TAGS ('{P}domain' = 'fraud', '{P}layer' = 'gold', '{P}contains_pii' = 'false', '{P}data_origin' = 'synthetic')")
print("commented and tagged")

In [0]:
import psycopg
r = spark.sql(f"""SELECT
  (SELECT COUNT(*) FROM {C}.silver_transfers) AS silver_rows,
  (SELECT COUNT(*) FROM {T1}) AS gold_rows,
  (SELECT COUNT(DISTINCT sequence_number) FROM {T1}) AS gold_distinct_seq,
  (SELECT SUM(flagged) FROM {T1}) AS flagged,
  (SELECT SUM(transfers) FROM {T2}) AS summary_transfers,
  (SELECT SUM(fraud_transfers) FROM {T2}) AS summary_fraud,
  (SELECT SUM(is_fraud) FROM {C}.silver_transfers) AS silver_fraud,
  (SELECT SUM(flagged_fraud_transfers) FROM {T2}) AS flagged_fraud""").first().asDict()
print(r)
assert r["silver_rows"] == r["gold_rows"] == r["gold_distinct_seq"] == r["summary_transfers"]
assert r["silver_fraud"] == r["summary_fraud"]
print(f"flagged share = {r['flagged'] / r['gold_rows']:.2%} | flagged-and-fraud = {r['flagged_fraud']} of {r['silver_fraud']} fraud (in-sample)")

# Independent check: scores for the same transfers in the Lakebase review queue (06_review_queue, same model and
# point-in-time method) should match gold scores.
LAKEBASE_EP = "projects/fraud-rt-online/branches/production/endpoints/primary"
host = w.postgres.get_endpoint(name=LAKEBASE_EP).status.hosts.host
token = w.postgres.generate_database_credential(endpoint=LAKEBASE_EP).token
with psycopg.connect(host=host, dbname="databricks_postgres", user=w.current_user.me().user_name, password=token, sslmode="require") as conn:
    q = pd.DataFrame(conn.execute("SELECT transfer_id, transfer_ts, fraud_score, model_version FROM fraud_review.review_queue").fetchall(),
                     columns=["transfer_id", "transfer_ts", "queue_score", "queue_model_version"])
q["transfer_ts"] = pd.to_datetime(q["transfer_ts"]).dt.tz_convert(None)
g = spark.table(T1).select("transfer_id", "ts", "fraud_score").toPandas()
m = q.merge(g, left_on=["transfer_id", "transfer_ts"], right_on=["transfer_id", "ts"], how="inner")
same_version = m[m.queue_model_version == str(MODEL_VERSION)]
print(f"queue rows={len(q)} matched to gold={len(m)} same model version={len(same_version)} "
      f"max |gold - queue| = {(same_version.fraud_score - same_version.queue_score).abs().max():.2e}")